In [1]:
from xgboost import XGBClassifier

In [2]:
import xgboost

print(xgboost.__version__)

2.1.4


In [3]:
brew --version


NameError: name 'brew' is not defined

In [4]:
brew install libomp

SyntaxError: invalid syntax (2760538136.py, line 1)

In [5]:
brew --prefix

NameError: name 'brew' is not defined

In [6]:
brew install libomp


SyntaxError: invalid syntax (1568185342.py, line 1)

In [1]:
from xgboost import XGBClassifier

import xgboost
print(xgboost.__version__)

2.1.4


In [2]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

model_xgb.fit(X_train, y_train)

NameError: name 'X_train' is not defined

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from xgboost import XGBClassifier

In [4]:
df = pd.read_csv("../data/raw/creditcard.csv")

print(df.shape)

(284807, 31)


In [5]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
model_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

model_xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [9]:
y_pred_xgb = model_xgb.predict(X_test)

In [10]:
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.92      0.81      0.86        98

    accuracy                           1.00     56962
   macro avg       0.96      0.90      0.93     56962
weighted avg       1.00      1.00      1.00     56962



In [11]:
y_probability_xgb = model_xgb.predict_proba(X_test)[:, 1]

roc_auc_xgb = roc_auc_score(y_test, y_probability_xgb)

print("ROC-AUC:", roc_auc_xgb)

ROC-AUC: 0.9816398668358733


In [12]:
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.92      0.81      0.86        98

    accuracy                           1.00     56962
   macro avg       0.96      0.90      0.93     56962
weighted avg       1.00      1.00      1.00     56962



In [13]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_xgb)

print(cm)

[[56857     7]
 [   19    79]]


In [14]:
from sklearn.metrics import average_precision_score

pr_auc = average_precision_score(y_test, y_probability_xgb)

print("PR-AUC:", pr_auc)

PR-AUC: 0.8800354252834914


In [15]:
fraud_count = y_train.value_counts()[1]
normal_count = y_train.value_counts()[0]

scale_pos_weight = normal_count / fraud_count

print("Normal transactions:", normal_count)
print("Fraud transactions:", fraud_count)
print("Scale Pos Weight:", scale_pos_weight)

Normal transactions: 227451
Fraud transactions: 394
Scale Pos Weight: 577.2868020304569


In [16]:
model_xgb_balanced = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss"
)

model_xgb_balanced.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [17]:
y_pred_balanced = model_xgb_balanced.predict(X_test)

y_probability_balanced = model_xgb_balanced.predict_proba(X_test)[:, 1]

In [18]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

print(classification_report(y_test, y_pred_balanced))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_balanced))

print("ROC-AUC:", roc_auc_score(y_test, y_probability_balanced))

print("PR-AUC:", average_precision_score(y_test, y_probability_balanced))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.85      0.83      0.84        98

    accuracy                           1.00     56962
   macro avg       0.93      0.91      0.92     56962
weighted avg       1.00      1.00      1.00     56962

Confusion Matrix:
[[56850    14]
 [   17    81]]
ROC-AUC: 0.9789838698563273
PR-AUC: 0.8767424155009337


In [19]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]

for threshold in thresholds:
    y_pred_threshold = (y_probability_xgb >= threshold).astype(int)

    precision = precision_score(y_test, y_pred_threshold)
    recall = recall_score(y_test, y_pred_threshold)
    f1 = f1_score(y_test, y_pred_threshold)

    print(
        f"Threshold: {threshold} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f} | "
        f"F1: {f1:.3f}"
    )

Threshold: 0.1 | Precision: 0.856 | Recall: 0.847 | F1: 0.851
Threshold: 0.2 | Precision: 0.874 | Recall: 0.847 | F1: 0.860
Threshold: 0.3 | Precision: 0.882 | Recall: 0.837 | F1: 0.859
Threshold: 0.4 | Precision: 0.901 | Recall: 0.837 | F1: 0.868
Threshold: 0.5 | Precision: 0.919 | Recall: 0.806 | F1: 0.859


In [20]:
from ml.risk_engine import calculate_risk

print(calculate_risk(0.12))
print(calculate_risk(0.48))
print(calculate_risk(0.91))

ModuleNotFoundError: No module named 'ml'

In [21]:
from ml.risk_engine import calculate_risk

print(calculate_risk(0.12))
print(calculate_risk(0.48))
print(calculate_risk(0.91))

ModuleNotFoundError: No module named 'ml'

In [22]:
import sys
sys.path.append("..")


In [23]:
from ml.risk_engine import calculate_risk

print(calculate_risk(0.12))
print(calculate_risk(0.48))
print(calculate_risk(0.91))

{'risk_score': 0.12, 'risk_level': 'LOW', 'decision': 'APPROVE'}
{'risk_score': 0.48, 'risk_level': 'MEDIUM', 'decision': 'REVIEW'}
{'risk_score': 0.91, 'risk_level': 'HIGH', 'decision': 'BLOCK'}


In [24]:
model_xgb

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [25]:
scaler


StandardScaler()

In [26]:
print(model_xgb)
print(scaler)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)
StandardScaler()


In [27]:
import joblib

joblib.dump(model_xgb, "../ml/model.pkl")
joblib.dump(scaler, "../ml/scaler.pkl")

print("Model and scaler saved successfully")

Model and scaler saved successfully


In [28]:
import os

print(os.path.exists("../ml/model.pkl"))
print(os.path.exists("../ml/scaler.pkl"))

True
True


In [29]:
import pandas as pd

df = pd.read_csv("../data/raw/creditcard.csv")

fraud_row = df[df["Class"] == 1].iloc[0]

features = fraud_row.drop("Class").tolist()

print(features)

[406.0, -2.3122265423263, 1.95199201064158, -1.60985073229769, 3.9979055875468, -0.522187864667764, -1.42654531920595, -2.53738730624579, 1.39165724829804, -2.77008927719433, -2.77227214465915, 3.20203320709635, -2.89990738849473, -0.595221881324605, -4.28925378244217, 0.389724120274487, -1.14074717980657, -2.83005567450437, -0.0168224681808257, 0.416955705037907, 0.126910559061474, 0.517232370861764, -0.0350493686052974, -0.465211076182388, 0.320198198514526, 0.0445191674731724, 0.177839798284401, 0.261145002567677, -0.143275874698919, 0.0]
